In [ ]:
# Declaring the imports
import pandas as pd
import numpy as np
import re
from collections import defaultdict

In [ ]:
# Paths to the CSV files
TRAINING_PATH = "./Data/Training.csv"
TESTING_PATH = "./Data/Testing.csv"

In [ ]:
# Loading the CSV file
def load_dataset_file(file_path):
    df = pd.read_csv(file_path)
    return df

In [ ]:
# Detect target and symptom columns
class Detect_Target_Symptom_Columns:
    def __init__(self, dataframe):
        self.df = dataframe
        self.target_column = None
        self.symptom_columns = None
        
    def detect_target_column(self):
        target_candidates = ["disease", "prognosis"]
        t_col = [column for column in self.df.columns if column.lower() in target_candidates]
        self.target_column = t_col[0]
        return self.target_column
        
    def detect_symptom_columns(self):
        if self.target_column is None:
            self.detect_target_column()
        s_cols = [columns for columns in self.df.columns if columns != self.target_column]
        self.symptom_columns = s_cols
        return self.symptom_columns

In [ ]:
# Data preprocessing
class Data_Preprocessing:
    def __init__(self, tr_dataframe, ts_dataframe):
        self.tr_df = tr_dataframe
        self.ts_df = ts_dataframe
        
    def remove_the_column_133(self):
        if "Unnamed: 133" in self.tr_df.columns:
            self.tr_df.drop("Unnamed: 133", axis = 1, inplace = True)
        return self.tr_df

    def remove_the_last_label_in_testing(self):
        self.ts_df = self.ts_df.drop_duplicates(subset = ["prognosis"], keep = "first")
        return self.ts_df

In [ ]:
# Encoding the symptom label
def encode_labels(raw_values):
    classes = list(dict.fromkeys(pd.Series(raw_values)))
    cls_to_int = {cls: i for i, cls in enumerate(classes)}
    int_values = np.array([cls_to_int[v] for v in raw_values], dtype = np.int64)
    return classes, cls_to_int, int_values

In [ ]:
# One hot encoding the labels
def one_hot_encoding(y_int, n_classes):
    y = np.zeros((y_int.shape[0], n_classes), dtype = np.float32)
    y[np.arange(y_int.shape[0]), y_int] = 1.0
    return y

In [ ]:
# Splitter for Stratified k fold cross validation
def stratified_splitter(x, y, n_splits, shuffle = True, random_state = None):
    if shuffle and random_state is not None:
        random_num_gen = np.random.RandomState(random_state)
        indices = random_num_gen.permutation(len(x))
    else:
        indices = np.arange(len(x))
    y = np.array(y)
    class_indices = defaultdict(list)
    for index in indices:
        class_indices[y[index]].append(index)
    folds = [[] for _ in range(n_splits)]
    for _, index in class_indices.items():
        index = np.array(index)
        if shuffle:
            random_num_gen.shuffle(index)
        fold_sizes = [len(index) // n_splits] * n_splits
        for i in range(len(index) % n_splits):
            fold_sizes[i] += 1
        start = 0
        for fold_index, size in enumerate(fold_sizes):
            folds[fold_index].extend(index[start:start + size])
            start += size
    for fold_index in range(n_splits):
        validation_fold = folds[fold_index]
        training_folds = [i for j in range(n_splits) if j != fold_index for i in folds[j]]
        yield validation_fold, training_folds

In [ ]:
# Single neuron class
class Neuron:
    def __init__(self, n_inputs, n_outputs, w_init):
        if w_init == "he":
            self.w = np.random.randn(n_inputs, n_outputs) * np.sqrt(2.0 / n_inputs)
        elif w_init == "xavier":
            self.w = np.random.randn(n_inputs, n_outputs) * np.sqrt(1.0 / n_inputs)
        self.b = np.zeros((1, n_outputs))
        self.dw = np.zeros_like(self.w)
        self.db = np.zeros_like(self.b)

    def forward_pass(self, x):
        self.x = x
        return np.dot(x, self.w) + self.b

    def backward_pass(self, dz):
        self.dw = np.dot(self.x.T, dz)
        self.db = np.sum(dz, axis = 0, keepdims = True)
        dx = np.dot(dz, self.w.T)
        return dx

    def step(self, lr):
        self.w -= lr * self.dw
        self.b -= lr * self.db

In [ ]:
# RelU activation
class ReLU_activation:
    def forward_pass(self, z):
        self.z = z
        return np.maximum(0.0, z)
    def backward_pass(self, dvalues_a):
        dz = dvalues_a.copy()
        dz[self.z <= 0.0] = 0
        return dz

In [ ]:
# Softmax activation
class Softmax_activation:
    def forward_pass(self, inputs):
        exp_values = np.exp(inputs - np.max(inputs, axis = 1, keepdims = True))
        self.probabilities = exp_values / np.sum(exp_values, axis = 1, keepdims = True)

In [ ]:
# Categorical cross entropy loss
class Categorical_Cross_Entropy_loss:
    def calculate_loss(self, p_values, t_values):
        samples = len(p_values)
        p_values_clipped = np.clip(p_values, 1e-7, 1 - 1e-7)
        if len(t_values.shape) == 1:
            correct_confidences = p_values_clipped[range(samples), t_values]
        elif len(t_values.shape) == 2:
            correct_confidences = np.sum(p_values_clipped * t_values, axis = 1)
        negative_log_likelihoods = -np.log(correct_confidences)
        return np.mean(negative_log_likelihoods)

In [ ]:
# Combined Softmax and Cross entropy loss
class Softmax_activation_Categorical_Cross_Entropy_loss:
    def __init__(self):
        self.activation = Softmax_activation()
        self.loss = Categorical_Cross_Entropy_loss()
    
    def forward_pass(self, inputs, t_values):
        self.activation.forward_pass(inputs)
        self.output = self.activation.probabilities
        return self.loss.calculate_loss(self.output, t_values)
    
    def backward_pass(self, d_values, t_values):
        samples = len(d_values)
        if len(t_values.shape) == 2:
            t_values = np.argmax(t_values, axis = 1)
        self.d_inputs = d_values.copy()
        self.d_inputs[range(samples), t_values] -= 1
        self.d_inputs = self.d_inputs / samples

In [ ]:
# Adam optimizer
class Adam_optimizer:
    def __init__(self, learning_rate, decay_rate, epsilon, beta_1, beta_2):
        self.learning_rate = learning_rate
        self.current_learning_rate = learning_rate
        self.decay_rate = decay_rate
        self.epsilon = epsilon
        self.beta_1 = beta_1
        self.beta_2 = beta_2
        self.iteration_number = 0

    def pre_update_params(self):
        if self.decay_rate:
            self.current_learning_rate = self.learning_rate * (1. / (1. + self.decay_rate * self.iteration_number))

    def update_params(self, layer):
        if not hasattr(layer, "weight_momentums"):
            layer.weight_momentums = np.zeros_like(layer.w)
            layer.weight_cache = np.zeros_like(layer.w)
            layer.bias_momentums = np.zeros_like(layer.b)
            layer.bias_cache = np.zeros_like(layer.b)
        layer.weight_momentums = self.beta_1 * layer.weight_momentums + (1 - self.beta_1) * layer.dw
        layer.bias_momentums = self.beta_1 * layer.bias_momentums + (1 - self.beta_1) * layer.db

        corrected_weight_momentums = layer.weight_momentums / (1 - self.beta_1 ** (self.iteration_number + 1))
        corrected_bias_momentums = layer.bias_momentums / (1 - self.beta_1 ** (self.iteration_number + 1))

        layer.weight_cache = self.beta_2 * layer.weight_cache + (1 - self.beta_2) * layer.dw ** 2
        layer.bias_cache = self.beta_2 * layer.bias_cache + (1 - self.beta_2) * layer.db ** 2

        corrected_weight_cache = layer.weight_cache / (1 - self.beta_2 ** (self.iteration_number + 1))
        corrected_bias_cache = layer.bias_cache / (1 - self.beta_2 ** (self.iteration_number + 1))

        layer.w += - self.current_learning_rate * corrected_weight_momentums / (np.sqrt(corrected_weight_cache) + self.epsilon)
        layer.b += - self.current_learning_rate * corrected_bias_momentums / (np.sqrt(corrected_bias_cache) + self.epsilon)

    def post_update_params(self):
        self.iteration_number += 1

In [ ]:
# Neuron dropout
class Dropout:
    def __init__(self, rate):
        self.rate = rate

    def forward_pass(self, x, training = True):
        if not training:
            return x
        self.mask = np.random.binomial(1, 1 - self.rate, size=x.shape) / (1 - self.rate)
        return x * self.mask
    
    def backward_pass(self, d_values):
        return d_values * self.mask

In [ ]:
# Core MLP
class MLP:
    def __init__(self, input_dim, hidden_1 = 64, hidden_2 = 32, n_classes = 10, learning_rate = 0.01, dropout_rate = 0.2):
        self.input_layer = Neuron(input_dim, hidden_1, w_init = "he")
        self.activation_1 = ReLU_activation()
        self.dropout_1 = Dropout(dropout_rate)
        self.hidden_layer_1 = Neuron(hidden_1, hidden_2, w_init = "he")
        self.activation_2 = ReLU_activation()
        self.dropout_2 = Dropout(dropout_rate)
        self.hidden_layer_2 = Neuron(hidden_2, n_classes, w_init = "xavier")
        self.softmax_loss = Softmax_activation_Categorical_Cross_Entropy_loss()
        self.learning_rate = learning_rate
        self.number_of_classes = n_classes

    def forward(self, x, y_true = None):
        z1 = self.input_layer.forward_pass(x)
        a1 = self.activation_1.forward_pass(z1)
        d_a1 = self.dropout_1.forward_pass(a1, True)
        z2 = self.hidden_layer_1.forward_pass(d_a1)
        a2 = self.activation_2.forward_pass(z2)
        d_a2 = self.dropout_2.forward_pass(a2, True)
        z3 = self.hidden_layer_2.forward_pass(d_a2)
        if y_true is not None:
            loss = self.softmax_loss.forward_pass(z3, y_true)
            return self.softmax_loss.output, loss
        self.softmax_loss.activation.forward_pass(z3)
        return self.softmax_loss.activation.probabilities

    def backward(self, y_true):
        self.softmax_loss.backward_pass(self.softmax_loss.output, y_true)
        da2 = self.hidden_layer_2.backward_pass(self.softmax_loss.d_inputs)
        dz2 = self.activation_2.backward_pass(da2)
        d_dz2 = self.dropout_2.backward_pass(dz2)
        da1 = self.hidden_layer_1.backward_pass(d_dz2)
        dz1 = self.activation_1.backward_pass(da1)
        d_dz1 = self.dropout_1.backward_pass(dz1)
        _ = self.input_layer.backward_pass(d_dz1)

    def w_step(self):
        for layer in [self.input_layer, self.hidden_layer_1, self.hidden_layer_2]:
            layer.step(self.learning_rate)

    def predict(self, x):
        probs = self.forward(x)
        return np.argmax(probs, axis = 1), probs

    def fit(self, x, y_one_hot, epochs = 50, batch_size = 64, x_val = None, y_val = None, optimizer = None):
        n = x.shape[0]
        for epoch in range(1, epochs + 1):
            epoch_loss = 0.0
            seen = 0
            for start in range(0, n, batch_size):
                end = min(start + batch_size, n)
                xb, yb = x[start : end], y_one_hot[start : end]
                probs, loss = self.forward(xb, yb)
                self.backward(yb)
                if optimizer is not None:
                    optimizer.pre_update_params()
                    for layer in [self.input_layer, self.hidden_layer_1, self.hidden_layer_2]:
                        optimizer.update_params(layer)
                    optimizer.post_update_params()
                else:
                    self.w_step()
                epoch_loss += loss * (end - start)
                seen += (end - start)
            epoch_loss /= seen
            if epoch == 1 or epoch % 5 == 0:
                msg = f"Epoch {epoch:3d} | loss {epoch_loss:.4f}"
                if x_val is not None and y_val is not None:
                    y_pred, _ = self.predict(x_val)
                    conf_matrix, _ = confusion_matrix(y_val, y_pred, self.number_of_classes)
                    correct_predictions = np.sum(np.diag(conf_matrix))
                    total_predictions = np.sum(conf_matrix)
                    val_acc = correct_predictions / total_predictions
                    msg += f" | val_acc {val_acc:.4f}"
                print(msg)

In [ ]:
# Confusion matrix
def confusion_matrix(y_true, y_pred, num_classes):
    conf_matrix = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        if p == -1:
            continue
        conf_matrix[t][p] += 1
    abstains = np.sum(y_pred == -1)
    return conf_matrix, abstains

In [ ]:
# Calculations for F1 score , accuracy
def calculation(confusion_matrix):
    true_positive = np.diag(confusion_matrix)
    false_positive = np.sum(confusion_matrix, axis=0) - true_positive
    false_negative = np.sum(confusion_matrix, axis=1) - true_positive
    precision = np.divide(true_positive, (true_positive + false_positive), out = np.zeros_like(true_positive, dtype = float), where = (true_positive + false_positive) != 0)
    recall = np.divide(true_positive, (true_positive + false_negative), out = np.zeros_like(true_positive, dtype = float), where = (true_positive + false_negative) != 0)
    f1_score = np.divide(2 * precision * recall, (precision + recall), out = np.zeros_like(precision, dtype = float), where = (precision + recall) != 0)
    return np.mean(f1_score), np.sum(true_positive) / np.sum(confusion_matrix)

In [ ]:
# Threshold tuner
def tune_threshold(y_true, y_prob, num_classes):
        max_probs = np.max(y_prob, axis=1)
        y_pred_standard = np.argmax(y_prob, axis=1)
        threshold_candidates = np.unique(max_probs)
        threshold_candidates = np.sort(np.unique(np.concatenate(([0.0, 1.0], threshold_candidates))))
        best_f1_score = -1.0
        best_threshold = 0.5
        for threshold in threshold_candidates:
            acceptance_mask = (max_probs >= threshold)
            if np.sum(acceptance_mask) == 0:
                continue
            y_true_accepted = y_true[acceptance_mask]
            y_pred_accepted = y_pred_standard[acceptance_mask]
            conf_matrix, _ = confusion_matrix(y_true_accepted, y_pred_accepted, num_classes)
            f1_score, _ = calculation(conf_matrix)
            if f1_score >= best_f1_score:
                best_f1_score = f1_score
                best_threshold = threshold
        return best_threshold, best_f1_score

In [ ]:
# Creating the input vector
class Conversation_and_Vector:
    def __init__(self, model, classes, symptom_index, threshold_gap):
        self.model = model
        self.classes = classes
        self.symptom_index = symptom_index
        self.threshold_gap = threshold_gap
        self.vector = np.zeros((1, len(symptom_index)), dtype=np.float32)

    def update_vector(self, user_input):
        tokens = [w.strip().lower() for w in re.split(r"[,.;]", user_input) if w.strip()]
        for t in tokens:
            if t in self.symptom_index:
                self.vector[0, self.symptom_index[t]] = 1.0

    def predict(self):
        _, probs = self.model.predict(self.vector)
        top_index_2 = np.argsort(probs[0])[-2:][::-1]
        top_conf, second_conf = probs[0][top_index_2[0]], probs[0][top_index_2[1]]
        top_disease, second_disease = self.classes[top_index_2[0]], self.classes[top_index_2[1]]
        return top_disease, top_conf, second_disease, second_conf
    
    def conversation_loop(self):
        while True:
            user_input = input().strip().lower()
            if not user_input:
                break
            self.update_vector(user_input)
            top_disease, top_conf, second_disease, second_conf = self.predict()
            if top_conf >= self.threshold_gap:
                print(f"Final prediction: { top_disease } with confidence: { top_conf }")
                break
            else:
                print(f"It could be { top_disease } confidence: { top_conf } or { second_disease } confidence: { second_conf }. Could you share more symptoms...")

In [ ]:
# The main method
def main():
    train_df = load_dataset_file(TRAINING_PATH)
    test_df = load_dataset_file(TESTING_PATH)
    preprocess = Data_Preprocessing(train_df, test_df)
    train_df = preprocess.remove_the_column_133()
    test_df = preprocess.remove_the_last_label_in_testing()
    detect_columns = Detect_Target_Symptom_Columns(train_df)
    target_column = detect_columns.detect_target_column()
    symptom_columns = detect_columns.detect_symptom_columns()
    x_train = train_df[symptom_columns].values.astype(np.float32)
    classes, cls_to_int, y_train_int = encode_labels(train_df[target_column])
    y_train_one_hot = one_hot_encoding(y_train_int, len(classes))
    x_test = test_df[symptom_columns].values.astype(np.float32)
    y_test_int = np.array([cls_to_int[v] for v in test_df[target_column].values], dtype = np.int64)
    optimal_thresholds = []
    fold_count = 0
    for validation_index, training_index in stratified_splitter(x_train, y_train_int, 5, True, 42):
        fold_count += 1
        x_val_fold, x_train_fold = x_train[validation_index], x_train[training_index]
        y_val_fold, y_train_fold = y_train_int[validation_index], y_train_one_hot[training_index]
        model = MLP(x_train_fold.shape[1], 64, 32, len(classes), 0.01)
        a_optimizer = Adam_optimizer(0.001, 1e-3, 1e-7, 0.9, 0.999)
        model.fit(x_train_fold, y_train_fold, 50, 64, a_optimizer)
        y_pred = model.forward(x_val_fold)
        best_threshold, best_f1_score = tune_threshold(y_val_fold, y_pred, len(classes))
        optimal_thresholds.append(best_threshold)
        print(f"Fold {fold_count}: Optimal Reject Threshold = {best_threshold:.4f}, Mean F1 (Accepted) = {best_f1_score:.4f}\n")
    final_optimal_threshold = np.mean(optimal_thresholds)
    print(f"\nAverage Optimal Reject Threshold from CV: {final_optimal_threshold:.4f}\n")
    f_model = MLP(x_train.shape[1], 64, 32, len(classes), 0.01)
    f_a_optimizer = Adam_optimizer(0.001, 1e-3, 1e-7, 0.9, 0.999)
    f_model.fit(x_train, y_train_one_hot, 100, 64, x_val = None, y_val = None, optimizer = f_a_optimizer)
    y_pred, _ = f_model.predict(x_test)
    print(f"\nTest accuracy: {(y_pred == y_test_int).mean():.4f}")
    symptom_index = {s.lower(): i for i, s in enumerate(symptom_columns)}
    print("\nEnter symptoms separated by commas (empty line to exit):")
    conversation_and_vector = Conversation_and_Vector(f_model, classes, symptom_index, final_optimal_threshold)
    conversation_and_vector.conversation_loop()

In [ ]:
# The main function call
main()

Epoch   1 | loss 3.7142
Epoch   5 | loss 3.4687
Epoch  10 | loss 3.1157
Epoch  15 | loss 2.6269
Epoch  20 | loss 2.0554
Epoch  25 | loss 1.5100
Epoch  30 | loss 1.1073
Epoch  35 | loss 0.8575
Epoch  40 | loss 0.6537
Epoch  45 | loss 0.5527
Epoch  50 | loss 0.4718
Fold 1: Optimal Reject Threshold = 0.8935, Mean F1 (Accepted) = 1.0000

Epoch   1 | loss 3.7646
Epoch   5 | loss 3.5063
Epoch  10 | loss 3.1781
Epoch  15 | loss 2.7404
Epoch  20 | loss 2.2333
Epoch  25 | loss 1.7132
Epoch  30 | loss 1.3147
Epoch  35 | loss 0.9796
Epoch  40 | loss 0.7661
Epoch  45 | loss 0.5939
Epoch  50 | loss 0.4928
Fold 2: Optimal Reject Threshold = 0.8004, Mean F1 (Accepted) = 1.0000

Epoch   1 | loss 3.7555
Epoch   5 | loss 3.5588
Epoch  10 | loss 3.2650
Epoch  15 | loss 2.8506
Epoch  20 | loss 2.3392
Epoch  25 | loss 1.8439
Epoch  30 | loss 1.4105
Epoch  35 | loss 1.0862
Epoch  40 | loss 0.8421
Epoch  45 | loss 0.6779
Epoch  50 | loss 0.5257
Fold 3: Optimal Reject Threshold = 0.7154, Mean F1 (Accepted) = 

 itching


It could be Fungal infection confidence: 0.5294899663728392 or Chronic cholestasis confidence: 0.09064671125141414. Could you share more symptoms...


 skin_rash


It could be Fungal infection confidence: 0.612844678584416 or Drug Reaction confidence: 0.21163180008836452. Could you share more symptoms...


 nodal_skin_eruptions


Final prediction: Fungal infection with confidence: 0.976020067655982
